# Real-Time Audio Emotion Prediction  
# (실시간 음성 감정 인식 추론)

This notebook records short voice clips, extracts MFCC features, scales them using a pre-trained scaler,  
and predicts the emotion using a pre-trained CNN model.  
본 노트북은 음성을 실시간으로 녹음하고, 사전 학습된 스케일러와 CNN 모델을 이용해 감정을 예측합니다.  


## 1. Import Libraries & Load Model / 라이브러리 및 모델 불러오기

- Loads the pre-trained **Keras model** (`audio_emotion_model.keras`)  
- Loads the previously saved **StandardScaler** (`scaler.save`)  
- These ensure consistent preprocessing and inference environment  
--- 
- 사전 학습된 **Keras 모델**(`audio_emotion_model.keras`)과  
- 정규화에 사용된 **StandardScaler**(`scaler.save`)를 불러옵니다.  
- 이를 통해 학습 시와 동일한 전처리 환경을 유지합니다.


In [ ]:
import sounddevice as sd
import numpy as np
import librosa
import joblib
from tensorflow.keras.models import load_model

# 1. 모델 & 스케일러 불러오기
model = load_model('audio_emotion_model.keras')
scaler = joblib.load('scaler.save')

## 2. Define Emotion Labels / 감정 라벨 정의

Defines the target emotion classes in the same order used during training:  
훈련 시 사용한 감정 클래스 순서로 라벨을 정의합니다.


In [ ]:
# 2. 감정 라벨 
emotion_labels = ['angry', 'happy', 'neutral', 'sad']

 ## 3. Audio Recording Function / 오디오 녹음 함수

- Records voice input via the system microphone using `sounddevice`  
- Default: **3 seconds** at **22,050 Hz** sampling rate  
- Returns the recorded audio signal and its sampling rate  
---
- `sounddevice` 라이브러리를 사용하여 마이크 입력을 녹음합니다.  
- 기본 설정은 **3초**, **22,050Hz**이며, 녹음된 오디오 신호와 샘플링 레이트를 반환합니다.


In [ ]:
# 3. 오디오 녹음 함수
def record_audio(duration=3, sr=22050):
    print(f"\n 말해주세요...")
    audio = sd.rec(int(duration * sr), samplerate=sr, channels=1)
    sd.wait()
    audio = audio.flatten()
    return audio, sr

## 4. MFCC Extraction & Scaling / MFCC 추출 및 정규화

- Extracts 100-dimensional **MFCC** features using `librosa`  
- Computes the mean across time frames  
- Applies scaling using the pre-trained `StandardScaler`  
- Expands the shape to `(100, 1)` for Conv1D model compatibility  
---

- `librosa`를 이용해 100차원 **MFCC 특징**을 추출하고,  
- 시간축 평균을 구한 후 저장된 `StandardScaler`로 정규화합니다.  
- 최종적으로 Conv1D 모델 입력 형태인 `(100, 1)`로 변환합니다.


In [ ]:
# 4. MFCC 추출 + 스케일링
def extract_features(audio, sr):
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=100)
    mfcc_mean = np.mean(mfcc.T, axis=0)
    mfcc_scaled = scaler.transform([mfcc_mean])
    mfcc_scaled = np.expand_dims(mfcc_scaled, axis=-1)  
    return mfcc_scaled

## 5. Emotion Prediction Function / 감정 예측 함수

- Calls `extract_features()` to preprocess audio  
- Reshapes features to match model input: `(1, 100, 1)`  
- Runs the model prediction and returns:
  - The predicted emotion label  
  - Confidence score (softmax probability)  
---
- 오디오에서 특징을 추출하고 `(1,100,1)` 형태로 변환한 후,  
- 모델을 통해 감정을 예측합니다. 예측된 감정 라벨과 신뢰도를 반환합니다.

In [ ]:
# 5. 예측 함수
def predict_emotion(audio, sr):
    features = extract_features(audio, sr)         # shape: (100,) 또는 (100,1)

    # 항상 (100, 1) 형태로 만들기
    features = np.reshape(features, (100, 1))       # (100,1)
    features = np.expand_dims(features, axis=0)     # (1,100,1) - 배치 차원 추가

    # 예측
    pred = model.predict(features)
    emotion_idx = np.argmax(pred)
    confidence = np.max(pred)
    return emotion_labels[emotion_idx], confidence

## 6. Main Execution Loop / 메인 실행 루프

- Continuously records and predicts emotions in real-time  
- Displays emotion label and confidence  
- Press **q** to quit  
---
- 실시간으로 음성을 녹음하고 감정을 예측합니다.  
- 결과로 예측된 감정과 신뢰도를 출력하며,  
- **q**를 입력하면 종료됩니다.


In [ ]:
# 6. 실행
if __name__ == "__main__":
    while True:
        audio, sr = record_audio()
        emotion, confidence = predict_emotion(audio, sr)
        print(f"예측된 감정: {emotion.upper()} (신뢰도: {confidence:.2%})")

        again = input("\n계속하려면 Enter, 종료하려면 q: ")
        if again.lower() == 'q':
            break



 말해주세요...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
예측된 감정: NEUTRAL (신뢰도: 99.99%)



계속하려면 Enter, 종료하려면 q:  q
